In [1]:
import os
print(os.getcwd())



/Users/devanshii/Documents/C22-veNTUre-project/Scripts


In [5]:
print(os.listdir("Data/raw/user_data/"))

FileNotFoundError: [Errno 2] No such file or directory: 'Data/raw/user_data/'

In [6]:
import os
print("Current folder:", os.getcwd())
print("Contents of current folder:", os.listdir("."))

Current folder: /Users/devanshii/Documents/C22-veNTUre-project/Scripts
Contents of current folder: ['master_ds.py', 'check.py', 'rebuilt_master_ds.py', 'check2.ipynb']


In [7]:
print(os.listdir("Data"))

FileNotFoundError: [Errno 2] No such file or directory: 'Data'

In [8]:
import os
print("I am here:", os.getcwd())
print("I can see:", os.listdir("."))

I am here: /Users/devanshii/Documents/C22-veNTUre-project/Scripts
I can see: ['master_ds.py', 'check.py', 'rebuilt_master_ds.py', 'check2.ipynb']


In [9]:
print(os.listdir("../Data"))

['.DS_Store', 'processed', 'Raw']


In [10]:
print(os.listdir("../Data/raw"))
print(os.listdir("../Data/raw/user_data"))

['.DS_Store', 'user_data', 'user_trades']
['Campaign 33 Data 24 Feb 2026 Traders only (1D).xlsx', 'Campaign 42 Data 31 Mar 2026 Traders only (1D).xlsx', 'Campaign 52 Data 15 May 2026 Traders only (1D).csv', 'Campaign 47 Data 24 Apr 2026 Traders only (1D).csv', 'Campaign 61 Data 16 June 2026 Traders only (1D).xlsx', 'Campaign 63 Data 23 June 2026 Traders only (1D).xlsx', 'Campaign 57 Data 02 June 2026 Traders only (1D).xlsx', 'Campaign 54 Data 22 May 2026 Traders only (1D).csv', 'Campaign 59 Data 09 Jun 2026 Traders only (1D).csv', 'Campaign 58 Data 05 June 2026 Traders only (1D).xlsx', 'Campaign 62 Data 19 June 2026 Traders only (1D).xlsx', 'Campaign 66 Data 03 July 2026 Traders only (1D).xlsx', 'Campaign 46 Data 17 Apr 2026 Traders only (1D).csv', 'Campaign 36 Data 09 Mar 2026 Traders only (1D).xlsx', 'Campaign 51 Data 12 May 2026 Traders only (1D).csv', 'Campaign 48 Data 28 Apr 2026 Traders only (1D).csv', 'Campaign 55 Data 26 May 2026 Traders only (1D).csv', 'Campaign 39 Data 20 Mar

In [12]:
import glob
folder = "../Data/raw/user_data/"
all_files = glob.glob(folder + "*.csv") + glob.glob(folder + "*.xlsx")

In [13]:
print(len(all_files))
print(all_files)

34
['../Data/raw/user_data/Campaign 52 Data 15 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 47 Data 24 Apr 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 54 Data 22 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 59 Data 09 Jun 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 46 Data 17 Apr 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 51 Data 12 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 48 Data 28 Apr 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 55 Data 26 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 45 Data 14 Apr 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 56 Data 29 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 50 Data 07 May 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 44 Data 10 Apr 2026 Traders only (1D).csv', '../Data/raw/user_data/Campaign 53 Data 19 May 2026 Traders only (1D).csv', '../Data

In [14]:
import pandas as pd
import re

dfs = []
for f in all_files:
    if f.endswith(".csv"):
        temp = pd.read_csv(f)
    else:
        temp = pd.read_excel(f)
    temp.columns = [c.strip().lower().replace(" ", "_") for c in temp.columns]
    match = re.search(r"Campaign (\d+)", f)
    temp["campaignId"] = match.group(1) if match else None
    temp["filename"] = f
    dfs.append(temp)

users_2_df = pd.concat(dfs, ignore_index=True)
print("Loaded rows:", len(users_2_df))
print("Missing account values:", users_2_df["account"].isna().sum())

Loaded rows: 15878
Missing account values: 0


In [15]:
print(users_2_df["campaignId"].value_counts().sort_index())

campaignId
33    500
34    281
35    500
36    401
37    455
38    500
39    340
40    381
41    223
42    500
43    500
44    500
45    500
46    500
47    500
48    500
49    500
50    500
51    500
52    500
53    500
54    500
55    500
56    500
57    504
58    298
59    500
60    500
61    500
62    495
63    500
64    500
65    500
66    500
Name: count, dtype: int64


In [1]:
import duckdb
con = duckdb.connect("../Data/processed/project.duckdb")
print(con.execute("SHOW TABLES").fetchdf())

                 name
0              trades
1  trades_with_trader
2               users


In [3]:
import duckdb
import pandas as pd
from scipy import stats

# ---------------------------------------------------------
# Step 0: Connect and load everything needed
# ---------------------------------------------------------
con = duckdb.connect("../Data/processed/project.duckdb")

trades_clean_df = con.execute("SELECT * FROM trades_clean").fetchdf()

account_summary = con.execute("""
    SELECT accountId,
           COUNT(*) AS n_trades,
           SUM(netProfit) AS total_netProfit
    FROM trades_clean
    GROUP BY accountId
""").fetchdf()
account_summary["outcome"] = account_summary["total_netProfit"].apply(lambda x: "winner" if x > 0 else "loser")

trades_with_trader_df = con.execute("SELECT * FROM trades_with_trader WHERE traderId IS NOT NULL").fetchdf()
rebuilt_account_summary = pd.read_csv("../Data/processed/rebuilt_account_summary.csv")

print("Loaded OLD account_summary:", account_summary.shape)
print("Loaded NEW rebuilt_account_summary:", rebuilt_account_summary.shape)

# ---------------------------------------------------------
# Part 1: Max drawdown — FAST vectorized version
# ---------------------------------------------------------
def fast_max_drawdown(df, id_col):
    df = df.sort_values([id_col, "openDateTime"]).copy()
    df["equity"] = df.groupby(id_col)["netProfit"].cumsum()
    df["running_max"] = df.groupby(id_col)["equity"].cummax()
    df["drawdown"] = df["equity"] - df["running_max"]
    return df.groupby(id_col)["drawdown"].min().reset_index(name="max_drawdown")

old_dd = fast_max_drawdown(trades_clean_df, "accountId")
account_summary = account_summary.merge(old_dd, on="accountId", how="left")

new_dd = fast_max_drawdown(trades_with_trader_df, "traderId")
rebuilt_account_summary = rebuilt_account_summary.merge(new_dd, on="traderId", how="left")

old_winners = account_summary[account_summary["outcome"] == "winner"]
old_losers = account_summary[account_summary["outcome"] == "loser"]
old_u, old_p = stats.mannwhitneyu(old_winners["max_drawdown"].dropna(), old_losers["max_drawdown"].dropna())

new_winners = rebuilt_account_summary[rebuilt_account_summary["outcome"] == "winner"]
new_losers = rebuilt_account_summary[rebuilt_account_summary["outcome"] == "loser"]
new_u, new_p = stats.mannwhitneyu(new_winners["max_drawdown"].dropna(), new_losers["max_drawdown"].dropna())

print("\n=== MAX DRAWDOWN: winner vs loser ===")
print(f"{'':20} {'OLD (accountId)':>20} {'NEW (traderId)':>20}")
print(f"{'Winner median':20} {old_winners['max_drawdown'].median():>20.1f} {new_winners['max_drawdown'].median():>20.1f}")
print(f"{'Loser median':20} {old_losers['max_drawdown'].median():>20.1f} {new_losers['max_drawdown'].median():>20.1f}")
print(f"{'p-value':20} {old_p:>20.6f} {new_p:>20.6f}")

# ---------------------------------------------------------
# Part 2: No-SL asymmetry (trade level)
# ---------------------------------------------------------
old_slcut = con.execute("""
    SELECT has_SL, AVG(netProfit) AS avg_profit, MEDIAN(netProfit) AS median_profit, MIN(netProfit) AS worst_loss
    FROM trades_clean GROUP BY has_SL
""").fetchdf()

new_slcut = con.execute("""
    SELECT has_SL, AVG(netProfit) AS avg_profit, MEDIAN(netProfit) AS median_profit, MIN(netProfit) AS worst_loss
    FROM trades_with_trader GROUP BY has_SL
""").fetchdf()

print("\n=== NO-SL vs HAS-SL (trade level) ===")
print("OLD:\n", old_slcut)
print("NEW:\n", new_slcut)

# ---------------------------------------------------------
# Part 3: Winner/loser counts
# ---------------------------------------------------------
print("\n=== WINNER/LOSER COUNTS ===")
print("OLD:", account_summary["outcome"].value_counts().to_dict(), f"(n={len(account_summary)})")
print("NEW:", rebuilt_account_summary["outcome"].value_counts().to_dict(), f"(n={len(rebuilt_account_summary)})")

IOException: IO Error: Could not set lock on file "/Users/devanshii/Documents/C22-veNTUre-project/Data/processed/project.duckdb": Conflicting lock is held in /Library/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python (PID 53996) by user devanshii. See also https://duckdb.org/docs/stable/connect/concurrency